In [24]:
# 1/6 라이브러리 및 경로 설정
import pandas as pd
import numpy as np
from scipy.spatial import KDTree
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import os

BASE_DIR = r'C:\Users\seohg\OneDrive\바탕 화면\seoul_map'
DATA_DIR = r'C:\Users\seohg\OneDrive\바탕 화면\2026\seoul datalob contest'

print('설정 완료')

설정 완료


In [25]:
# 2/6 브랜드 생존 라벨링
brand_map = {
    '투썸':'투썸플레이스','이디야':'이디야','빽다방':'빽다방',
    '공차':'공차','할리스':'할리스','파스쿠찌':'파스쿠찌',
    '카페베네':'카페베네','엔제리너스':'엔제리너스','탐앤탐':'탐앤탐스',
    '폴바셋':'폴바셋','컴포즈':'컴포즈','메가커피':'메가커피','요거프레소':'요거프레소'
}

def get_brand(name):
    for kw, brand in brand_map.items():
        if kw in str(name):
            return brand
    return None

df = pd.read_csv(os.path.join(BASE_DIR, 'brand_history.csv'))
df['brand'] = df['상호명'].apply(get_brand)
df = df[df['brand'].notna()]
df['period_num'] = df['period'].str.replace('df_', '', regex=False).astype(int)

# 상가업소번호 기준 생존기간 계산
survival = df.groupby(['상가업소번호','brand']).agg(
    생존분기수=('period_num','nunique'),
    입점분기=('period_num','min'),
    층정보=('층정보','first'),
    건물관리번호=('건물관리번호','first'),
    경도=('경도','first'),
    위도=('위도','first'),
    시군구명=('시군구명','first'),
    행정동명=('행정동명','first'),
    도로명주소=('도로명주소','first')
).reset_index()

# 생존 라벨: 8분기(2년) 이상 = 1(생존), 미만 = 0(폐업)
survival['생존여부'] = (survival['생존분기수'] >= 8).astype(int)
survival['위치ID'] = survival['건물관리번호'].astype(str) + '_' + survival['층정보'].astype(str) + '_nan'

print(f'총 브랜드 위치: {len(survival):,}개')
print(f'생존율: {survival["생존여부"].mean():.1%}')
print()
print('브랜드별 생존율:')
print(survival.groupby('brand')['생존여부'].agg(['mean','count']).round(3).sort_values('mean', ascending=False))

총 브랜드 위치: 4,898개
생존율: 68.3%

브랜드별 생존율:
         mean  count
brand               
이디야     0.771   1324
메가커피    0.765     85
엔제리너스   0.764    106
공차      0.732    298
파스쿠찌    0.731    167
할리스     0.700    237
탐앤탐스    0.691    230
빽다방     0.678    593
요거프레소   0.665    236
폴바셋     0.635     52
카페베네    0.609     87
컴포즈     0.598    731
투썸플레이스  0.573    752


In [26]:
# 3/6 물리적 특성 + 상권 지표 결합
# viz_map_v2: 층구분, 연면적, 지하철거리, 건물내위치수, TRDAR_SE_1, 도로유형
# result_final: 공실횟수, 교체횟수, 상권평균공실, 상권평균교체, 위험점수

viz = pd.read_csv(
    os.path.join(BASE_DIR, 'viz_map_v2.csv'),
    usecols=['위치ID','층구분','연면적','지하철거리_m','건물내위치수','TRDAR_SE_1','도로유형','위험확률']
)

result = pd.read_csv(
    os.path.join(DATA_DIR, 'result_final.csv'),
    usecols=['위치ID','공실횟수','교체횟수','상권평균공실','상권평균교체','위험점수','위도','경도']
)

merged = survival.merge(viz, on='위치ID', how='left')
merged = merged.merge(result, on='위치ID', how='left')

print(f'물리적 특성 매칭률: {merged["층구분"].notna().mean():.1%}')
print(f'result_final 매칭률: {merged["공실횟수"].notna().mean():.1%}')

물리적 특성 매칭률: 62.1%
result_final 매칭률: 62.1%


In [27]:
# 4/6 주변 브랜드 수 / 주변 카페 수 계산 (KDTree)
# 브랜드 입점 자리 기준
# - 반경 500m 내 동일 브랜드 수 (경쟁 강도)
# - 반경 200m 내 전체 카페 수 (카테고리 경쟁)

# 좌표 있는 브랜드만
brand_loc = merged.dropna(subset=['경도_x','위도_x'])[['상가업소번호','brand','경도_x','위도_x']].copy()
brand_loc.columns = ['상가업소번호','brand','경도','위도']

# 전체 카페 위치 (brand_history 전체)
cafe_coords = df.groupby('상가업소번호').agg(경도=('경도','first'), 위도=('위도','first')).dropna()

cafe_tree = KDTree(cafe_coords[['위도','경도']].values)

# 반경 200m ≈ 위도 기준 0.0018도
def count_nearby(lat, lng, tree, radius_deg):
    return len(tree.query_ball_point([lat, lng], radius_deg)) - 1  # 자기 자신 제외

brand_loc['주변카페수_200m'] = brand_loc.apply(
    lambda r: count_nearby(r['위도'], r['경도'], cafe_tree, 0.0018), axis=1
)

# 브랜드별 동일 브랜드 수
same_brand_counts = {}
for brand_name, grp in brand_loc.groupby('brand'):
    coords = grp[['위도','경도']].values
    tree = KDTree(coords)
    counts = [len(tree.query_ball_point(c, 0.0045)) - 1 for c in coords]  # 500m
    same_brand_counts.update(dict(zip(grp['상가업소번호'], counts)))

brand_loc['동일브랜드수_500m'] = brand_loc['상가업소번호'].map(same_brand_counts)

merged = merged.merge(
    brand_loc[['상가업소번호','주변카페수_200m','동일브랜드수_500m']],
    on='상가업소번호', how='left'
)

print('주변 카페/브랜드 수 계산 완료')
print(merged[['주변카페수_200m','동일브랜드수_500m']].describe().round(1))

주변 카페/브랜드 수 계산 완료
       주변카페수_200m  동일브랜드수_500m
count     12017.0      12017.0
mean          5.9         19.5
std           4.0         22.6
min           0.0          0.0
25%           3.0          2.0
50%           5.0          8.0
75%           9.0         31.0
max          20.0         87.0


In [28]:
# 5/6 CatBoost 생존 모델 (변수 추가 버전)

cat_features = ['층구분','TRDAR_SE_1','도로유형']
num_features = [
    '연면적','지하철거리_m','건물내위치수',   # 물리적 특성
    '공실횟수','교체횟수',                    # 상권 활성도
    '상권평균공실','상권평균교체',             # 상권 평균
    '주변카페수_200m','동일브랜드수_500m'     # 주변 경쟁
]
features = cat_features + num_features

model_df = merged.dropna(subset=['층구분']).copy()
X = model_df[features].fillna(-1)
y = model_df['생존여부']

print(f'학습 데이터: {len(model_df):,}개')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=500, learning_rate=0.05,
    cat_features=cat_features,
    verbose=0, random_seed=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print(f'\nROC-AUC: {roc_auc_score(y_test, y_prob):.3f}')
print()
print(classification_report(y_test, y_pred))
print()
print('=== Feature Importance ===')
fi = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
for k, v in fi.items():
    print(f'  {k}: {v:.1f}%')

학습 데이터: 9,859개

ROC-AUC: 0.943

              precision    recall  f1-score   support

           0       0.90      0.74      0.81       751
           1       0.86      0.95      0.90      1221

    accuracy                           0.87      1972
   macro avg       0.88      0.85      0.86      1972
weighted avg       0.87      0.87      0.87      1972


=== Feature Importance ===
  동일브랜드수_500m: 17.3%
  연면적: 13.3%
  공실횟수: 12.7%
  지하철거리_m: 12.3%
  교체횟수: 11.9%
  주변카페수_200m: 9.9%
  건물내위치수: 7.8%
  층구분: 6.7%
  도로유형: 3.9%
  상권평균공실: 2.2%
  상권평균교체: 1.1%
  TRDAR_SE_1: 0.7%


In [33]:
# 6/6 빈 자리 찾기 + 생존 확률 예측
# 마지막 분기(202503)에 공실인 위치 = 지금 비어있는 자리

# viz_map에서 마지막 분기 공실 위치
# result_final에서 공실횟수가 높고 교체횟수가 낮은 자리 = 오래 비어있는 자리
# 현재 공실 위치 (pivot_df 마지막 분기 기준)

pivot = pd.read_csv(
    os.path.join(DATA_DIR, 'pivot_df.csv'),
    index_col='위치ID',
    usecols=['위치ID', 'df_202512']
)
empty_ids = pivot[pivot['df_202512'].isna()].index.tolist()
print(f'현재 공실 위치: {len(empty_ids):,}개')

# result_final에서 해당 위치만
empty = result[result['위치ID'].isin(empty_ids)].copy()

# 중복 제거 (위치ID 기준)
empty = empty.drop_duplicates('위치ID').reset_index(drop=True)
print(f'중복 제거 후: {len(empty):,}개')

# 물리적 특성 결합
empty = empty.merge(viz, on='위치ID', how='left').dropna(subset=['층구분'])
print(f'물리적 특성 매칭 후: {len(empty):,}개')

# 주변 카페 수 계산
empty['주변카페수_200m'] = empty.apply(
    lambda r: len(cafe_tree.query_ball_point([r['위도'], r['경도']], 0.0018))-1
    if pd.notna(r.get('위도')) else -1, axis=1
)
empty['동일브랜드수_500m'] = 0

# 생존 확률 예측
X_empty = empty[features].fillna(-1)
empty['생존확률'] = model.predict_proba(X_empty)[:,1]

print()
print('=== 생존 확률 상위 20개 빈 자리 ===')
top = empty.sort_values('생존확률', ascending=False).head(20)
print(top[['위치ID','층구분','TRDAR_SE_1','지하철거리_m','도로명주소','생존확률']].to_string())

empty.to_csv(os.path.join(BASE_DIR, 'empty_locations.csv'), index=False, encoding='utf-8-sig')
print('\n✅ empty_locations.csv 저장 완료')

현재 공실 위치: 115,096개
중복 제거 후: 90,771개
물리적 특성 매칭 후: 99,474개

=== 생존 확률 상위 20개 빈 자리 ===
                                    위치ID    층구분 TRDAR_SE_1  지하철거리_m           도로명주소      생존확률
19566  1117013000103030017008406_nan_nan  층정보없음       골목상권    484.8    용산구 회나무로6길 5  0.989764
57528  1150010200105210022025934_nan_nan  층정보없음       골목상권    647.2   강서구 등촌로39길 56  0.989312
65314  1154510300108600051004988_nan_nan  층정보없음       골목상권   1122.3   금천구 독산로 157-1  0.988189
47722  1141011600100900049032897_nan_nan  층정보없음       골목상권    529.0  서대문구 신촌로7안길 80  0.988056
44197  1138010400105180014030167_nan_nan  층정보없음       골목상권    847.9   은평구 갈현로23길 34  0.987625
63819  1154510100102350038015239_nan_nan  층정보없음       골목상권    706.7    금천구 두산로9가길 7  0.987412
37742    1130510200102690002014748_4_nan   2층이상       골목상권   1487.1  강북구 한천로109길 13  0.987410
47874    1141011700100810031019717_1_nan     1층       골목상권   1537.3    서대문구 연희로 165  0.987197
15747  1117010200100160000024778_nan_nan  층정보없음       골목상권   1225.5   

In [30]:
print('train데이터 모델 평가',model.score(X_train,y_train))
print('test데이터 모델 평가',model.score(X_test,y_test))

train데이터 모델 평가 0.899581589958159
test데이터 모델 평가 0.8711967545638946
